# ARM 1 — Chunking Inspection\n**PDF:** `2004_05_27_2004A27101.pdf` &nbsp;|&nbsp; **Question:** 38\n\nRuns extraction → article location → chunking inline. No pre-generated files needed."

## 0 — Setup"

In [14]:
import json, sqlite3, sys
from pathlib import Path

import fitz  # PyMuPDF
import pandas as pd

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 50)

# ── Paths ─────────────────────────────────────────────────────────────────────
_HERE = Path().resolve()                          # notebook directory
_REPO = _HERE.parent

for _s in _REPO.glob("RQ2_T0*/src"):
    if str(_s) not in sys.path:
        sys.path.insert(0, str(_s))

from arm1_naive.chunker import chunk_recursive, chunk_sliding_window, locate_articles
from transformers import AutoTokenizer

DB_PATH        = Path("$RQ2_DATA_DIR/bsard_corpus.db")
PDF_DIR        = _REPO / "RQ2_T02_DATA_LOADER/data/pdfs"
TOKENIZER_NAME = "intfloat/multilingual-e5-large-instruct"

# ── Fixed targets ─────────────────────────────────────────────────────────────
PDF_NAME    = "2004_05_27_2004A27101.pdf"
QUESTION_ID = 38
WINDOW_SIZE = 512
STRIDE      = 256
MAX_TOKENS  = 512

print("Setup OK")

Setup OK


## 1 — Extract PDF text & locate articles

In [15]:
pdf_path = PDF_DIR / PDF_NAME
fitz_doc = fitz.open(str(pdf_path))
raw_text = "\n\n".join(page.get_text() for page in fitz_doc)
print(f"{len(fitz_doc)} pages  |  {len(raw_text):,} chars")

conn = sqlite3.connect(str(DB_PATH))
conn.row_factory = sqlite3.Row

article_spans = locate_articles(raw_text, PDF_NAME, conn)
print(f"{len(article_spans)} article spans found")

116 pages  |  551,233 chars
441 article spans found


## 2 — Question 38 & ground-truth articles

In [16]:
q_row = conn.execute(
    "SELECT question_text, relevant_article_ids FROM questions WHERE question_id = ?",
    (QUESTION_ID,),
).fetchone()

q_text         = q_row["question_text"]
gt_article_ids = json.loads(q_row["relevant_article_ids"])
gt_set         = set(gt_article_ids)

print(f"Q{QUESTION_ID}: {q_text}")
print(f"GT article_ids: {gt_article_ids}")

Q38: Quelle procédure en cas de défaut de paiement de ma facture d'eau en Wallonie ?
GT article_ids: [36353, 36354, 36355, 17011]


In [17]:
sep1 = "═" * 70
sep2 = "─" * 70

ph = ",".join("?" * len(gt_article_ids))
art_rows = conn.execute(
    f"SELECT article_id, article_number, article_text FROM articles WHERE article_id IN ({ph})",
    gt_article_ids,
).fetchall()
conn.close()

gt_articles = {r["article_id"]: dict(r) for r in art_rows}

for aid, art in gt_articles.items():
    print(f"\n{sep1}")
    print(f"article_id={aid}  |  article_number={art['article_number']}")
    print(sep2)
    print(art["article_text"])


══════════════════════════════════════════════════════════════════════
article_id=17011  |  article_number=D202
──────────────────────────────────────────────────────────────────────
La distribution publique d'eau à un immeuble affecté en tout ou en partie à l'habitation ne peut être interrompue :- que pour protéger la santé publique, la salubrité ou la continuité du service;- qu'à la demande de l'usager;- qu'en exécution d'une décision judiciaire rendue pour non-paiement et autorisant le recours à l'interruption de la distribution;- qu'en cas d'empêchement dûment constaté d'accéder au compteur, conformément à l'article D.207.La distribution publique d'eau à un immeuble qui n'est pas affecté à l'habitation ne peut être interrompue :- que dans les cas prévus par ou en vertu du décret;- qu'à la demande de l'usager;- qu'en cas de non-paiement après mise en demeure;- qu'en cas d'empêchement dûment constaté d'accéder au compteur, conformément à l'article 207.Lorsque le service est interrom

## 3 — Article spans detected in the PDF

In [18]:
spans_df = pd.DataFrame([
    {
        "article_id":  s.article_id,
        "start_char":  s.start_char,
        "end_char":    s.end_char,
        "char_length": s.end_char - s.start_char,
        "text_preview": raw_text[s.start_char : s.start_char + 120].replace("\n", " ").strip(),
    }
    for s in article_spans
])

def _hl_gt(row):
    color = "background-color: #c6efce" if row["article_id"] in gt_set else ""
    return [color] * len(row)

gt_spans = spans_df[spans_df["article_id"].isin(gt_set)]
print(f"{len(spans_df)} spans total | {len(gt_spans)} GT spans (highlighted)")
display(gt_spans.style.apply(_hl_gt, axis=1))

441 spans total | 1 GT spans (highlighted)


,article_id,start_char,end_char,char_length,text_preview
389,17011,456925,463810,6885,Art. D202.[1 § 1er. Le fonctionnaire sanctionnateur peut [2 recourir à]2 une procédure de médiation organisée par un m


## 4 — Chunk (runs tokenizer — takes ~30 s first time)

In [19]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
doc_id    = Path(PDF_NAME).stem

sw_chunks  = chunk_sliding_window(raw_text, tokenizer, doc_id, article_spans,
                                   window_size=WINDOW_SIZE, stride=STRIDE)
rec_chunks = chunk_recursive(raw_text, tokenizer, doc_id, article_spans,
                              max_tokens=MAX_TOKENS)

print(f"Sliding-window : {len(sw_chunks)} chunks")
print(f"Recursive      : {len(rec_chunks)} chunks")

Token indices sequence length is longer than the specified maximum sequence length for this model (141191 > 512). Running this sequence through the model will result in indexing errors


Sliding-window : 551 chunks
Recursive      : 347 chunks


In [20]:
# Build DataFrames for both strategies
def chunks_to_df(chunks):
    return pd.DataFrame([
        {
            "index":        i,
            "chunk_id":     c.chunk_id,
            "start_token":  c.start_token,
            "end_token":    c.end_token,
            "token_count":  c.end_token - c.start_token,
            "start_char":   c.start_char,
            "end_char":     c.end_char,
            "char_count":   c.end_char - c.start_char,
            "article_ids":  c.article_ids,
            "n_articles":   len(c.article_ids),
            "gt_hit":       bool(set(c.article_ids) & gt_set),
            "text_preview": c.text[:200].replace("\n", " ").strip(),
            "text_full":    c.text,
        }
        for i, c in enumerate(chunks)
    ])

sw  = chunks_to_df(sw_chunks)
rec = chunks_to_df(rec_chunks)
print("DataFrames ready")

DataFrames ready


## 5 — Strategy comparison

In [21]:
pd.DataFrame([
    {
        "strategy":      label,
        "total_chunks":  len(df),
        "gt_hit_chunks": int(df["gt_hit"].sum()),
        "unassigned":    int((df["n_articles"] == 0).sum()),
        "tok_min":       int(df["token_count"].min()),
        "tok_max":       int(df["token_count"].max()),
        "tok_mean":      round(df["token_count"].mean(), 1),
        "tok_std":       round(df["token_count"].std(), 1),
    }
    for label, df in [("sliding_window", sw), ("recursive", rec)]
]).set_index("strategy")

,total_chunks,gt_hit_chunks,unassigned,tok_min,tok_max,tok_mean,tok_std
strategy,,,,,,,
sliding_window,551,9,0,391,512,511.8,5.2
recursive,347,4,0,23,1165,406.9,133.0


## 6 — Chunks that hit Q38 ground-truth articles

In [22]:
COLS = ["index", "chunk_id", "token_count", "start_char", "end_char",
        "article_ids", "gt_hit", "text_preview"]

for label, df in [("Sliding-Window", sw), ("Recursive", rec)]:
    hits = df[df["gt_hit"]][COLS]
    print(f"\n── {label}: {len(hits)} / {len(df)} chunks hit GT articles {gt_article_ids} ──")
    display(hits)


── Sliding-Window: 9 / 551 chunks hit GT articles [36353, 36354, 36355, 17011] ──


,index,chunk_id,token_count,start_char,end_char,article_ids,gt_hit,text_preview
456,456,2004_05_27_2004A27101_sw_456,512,455008,457043,"[17010, 17011]",True,"peut, soit d'office, soit sur demande de la personne désignée par le Gouvernement, soit sur demande du collège communal de la commune sur le territoire de laquelle l'infraction a été commise, pron..."
457,457,2004_05_27_2004A27101_sw_457,512,456049,458135,"[17010, 17011]",True,"ation de toute exploitation ou toute partie d'exploitation, pendant une période d'un mois à cinq ans, à l'endroit où l'infraction a été commise ; 2° la fermeture, pour une période d'un mois à t..."
458,458,2004_05_27_2004A27101_sw_458,512,457044,459279,[17011],True,médiateur habilité pour traiter les dossiers en matière de sanctions administratives. Le Gouvernement détermine les conditions d'habilitation des médiateurs. La médiation correspond à une mesur...
459,459,2004_05_27_2004A27101_sw_459,512,458136,460378,[17011],True,"ou du refus du contrevenant à participer à cette procédure de médiation. A défaut de réponse endéans ce délai, le contrevenant est réputé avoir refusé la proposition.]2 § 2. [2 Lorsque le contr..."
460,460,2004_05_27_2004A27101_sw_460,512,459283,461447,[17011],True,"§ 3. En toute impartialité, le médiateur s'entretient avec le contrevenant [2 , les personnes éventuellement désignées conformément au paragraphe 2]2 et les victimes éventuelles des faits infracti..."
461,461,2004_05_27_2004A27101_sw_461,512,460378,462491,[17011],True,"2025 de médiation ainsi que de la convention signée. Lorsqu'il refuse l'homologation de la convention, le fonctionnaire sanctionnateur peut adresser ses remarques au médiateur afin que la conven..."
462,462,2004_05_27_2004A27101_sw_462,512,461447,463600,[17011],True,"échec de la procédure de médiation au cours de celle-ci ou lorsque le fonctionnaire sanctionnateur refuse l'homologation de la convention signée ou constate l'échec de la procédure de médiation]2, le"
463,463,2004_05_27_2004A27101_sw_463,512,462492,464626,"[17011, 17012]",True,"cadre de la procédure de médiation sont confidentiels, à l'exception de ce que les parties consentent à porter à la connaissance du fonctionnaire sanctionnateur. Ils ne peuvent être utilisés dans une"
464,464,2004_05_27_2004A27101_sw_464,512,463600,465620,"[17011, 17012]",True,"<DRW 2021-11-24/09, art. 55, 045; En vigueur : 01-07-2022> Section 2. [1 - Prestation citoyenne pour les majeurs]1 ---------- (1)<Inséré par DRW 2019-05-06/14, art. 1, 044; En vigueur : 01-..."



── Recursive: 4 / 347 chunks hit GT articles [36353, 36354, 36355, 17011] ──


,index,chunk_id,token_count,start_char,end_char,article_ids,gt_hit,text_preview
290,290,2004_05_27_2004A27101_rec_290,489,456500,458533,"[17010, 17011]",True,"Le fonctionnaire sanctionnateur peut compléter les mesures de restitution prononcées par des mesures complémentaires ou compensatoires au sens de l'article D.94, alinéa 1er, 13° et 14°. Dans sa..."
291,291,2004_05_27_2004A27101_rec_291,422,458533,460382,[17011],True,"Le cas échéant, le fonctionnaire sanctionnateur fixe les objectifs à atteindre dans le cadre de la procédure de médiation en matière de mesure de restitution. Dans un délai de dix jours à compter ..."
292,292,2004_05_27_2004A27101_rec_292,500,460385,462433,[17011],True,"de médiation ainsi que de la convention signée. Lorsqu'il refuse l'homologation de la convention, le fonctionnaire sanctionnateur peut adresser ses remarques au médiateur afin que la convention, e..."
293,293,2004_05_27_2004A27101_rec_293,435,462433,464241,"[17011, 17012]",True,"Les documents établis et les communications faites dans le cadre de la procédure de médiation sont confidentiels, à l'exception de ce que les parties consentent à porter à la connaissance du fonct..."


## 7 — Read full text of any chunk\nChange `STRATEGY` and `CHUNK_INDEX` to inspect any chunk.

In [23]:
STRATEGY    = sw   # sw  or  rec
CHUNK_INDEX = int(sw[sw["gt_hit"]].iloc[0]["index"])   # first GT-hit chunk; change freely

row = STRATEGY[STRATEGY["index"] == CHUNK_INDEX].iloc[0]
print(f"chunk_id    : {row['chunk_id']}")
print(f"tokens      : {row['token_count']}  ({row['start_token']}–{row['end_token']})")
print(f"chars       : {row['start_char']:,}–{row['end_char']:,}")
print(f"article_ids : {row['article_ids']}")
print(f"gt_hit      : {row['gt_hit']}")
print(f"\n{'─'*60}\n")
print(row["text_full"])

chunk_id    : 2004_05_27_2004A27101_sw_456
tokens      : 512  (116736–117248)
chars       : 455,008–457,043
article_ids : [17010, 17011]
gt_hit      : True

────────────────────────────────────────────────────────────

peut, soit d'office, soit sur
demande de la personne désignée par le Gouvernement, soit sur demande du collège communal de la commune
sur le territoire de laquelle l'infraction a été commise, prononcer, aux frais du contrevenant, les mesures de
restitutions suivantes :
   1° la remise en état ;
   2° la mise en oeuvre de mesures visant à faire cesser l'infraction ;
   3° l'exécution de mesures de nature à protéger la population ou l'environnement des nuisances causées ou de
mesures visant à empêcher l'accès aux lieux de l'infraction ;
   4° l'exécution de mesures de nature à atténuer les nuisances causées et ces conséquences ;
   5° l'exécution de travaux d'aménagement visant à régler la situation de manière transitoire avant la remise en
état ;
   6° la réalisation d'un

## 8 — All chunks covering a specific article\nChange `TARGET_ARTICLE_ID` to any id from the spans table.

In [24]:
TARGET_ARTICLE_ID = gt_article_ids[0]   # change as needed

for label, df in [("Sliding-Window", sw), ("Recursive", rec)]:
    mask   = df["article_ids"].apply(lambda ids: TARGET_ARTICLE_ID in ids)
    subset = df[mask][COLS]
    print(f"\n── {label}: {len(subset)} chunk(s) cover article {TARGET_ARTICLE_ID} ──")
    display(subset)


── Sliding-Window: 0 chunk(s) cover article 36353 ──


,index,chunk_id,token_count,start_char,end_char,article_ids,gt_hit,text_preview



── Recursive: 0 chunk(s) cover article 36353 ──


,index,chunk_id,token_count,start_char,end_char,article_ids,gt_hit,text_preview


## 9 — Unassigned chunks (no article_id matched)

In [25]:
for label, df in [("Sliding-Window", sw), ("Recursive", rec)]:
    unassigned = df[df["n_articles"] == 0][COLS]
    print(f"\n── {label}: {len(unassigned)} / {len(df)} chunks unassigned ──")
    display(unassigned)


── Sliding-Window: 0 / 551 chunks unassigned ──


,index,chunk_id,token_count,start_char,end_char,article_ids,gt_hit,text_preview



── Recursive: 0 / 347 chunks unassigned ──


,index,chunk_id,token_count,start_char,end_char,article_ids,gt_hit,text_preview
